# PT Monitor & Learning

Loads yesterday's race verdicts + dividends, calculates P&L, generates learnings via Claude, and updates the learning database.

**Cells:**
- 3a: Imports & config
- 3b: Load dividends + race verdict JSONs, join on raceId/saddle
- 3c: Calculate P&L → append to PT/betsMonitor.csv
- 3d: Generate learnings via Claude API
- 3e: Merge into PT/learnings_db.json


In [ ]:
# papermill parameters — injected by PT_WORKFLOW.ipynb when run automatically
ANTHROPIC_API_KEY = None


## 3a. Imports & Config

In [ ]:
# Standard library
import os, json, re, time
from datetime import datetime, timedelta, date

import numpy as np
import pandas as pd
import requests

!pip install -q anthropic
import anthropic

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
BASE  = '/content/drive/MyDrive/PT'
TODAY = date.today().strftime('%Y-%m-%d')
YESTERDAY = (date.today() - timedelta(days=1)).strftime('%Y-%m-%d')

# Override YESTERDAY if needed
# YESTERDAY = '2026-04-23'

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
# Or set directly:
# ANTHROPIC_API_KEY = 'sk-ant-...'

# ── Load system prompts from repo prompts/ dir (Railway) or GitHub raw (Colab)
_PROMPTS_LOCAL = os.path.join(os.getcwd(), 'prompts')
if not os.path.isdir(_PROMPTS_LOCAL):
    _PROMPTS_LOCAL = os.path.join(os.path.dirname(os.getcwd()), 'prompts')

def _load_prompt(filename):
    if os.path.isdir(_PROMPTS_LOCAL):
        with open(os.path.join(_PROMPTS_LOCAL, filename), encoding='utf-8') as _f:
            return _f.read()
    _token = os.environ.get('GITHUB_TOKEN', '')
    _url   = f'https://raw.githubusercontent.com/arauch6363-crypto/ptnew/main/prompts/{filename}'
    _r     = requests.get(_url, headers={'Authorization': f'token {_token}'} if _token else {})
    _r.raise_for_status()
    print(f'Loaded {filename} from GitHub')
    return _r.text

_LEARNING_SYSTEM_PROMPT  = _load_prompt('learning_system_prompt.txt')
_LEARNING_CURATOR_PROMPT = _load_prompt('learning_curator_prompt.txt')

print(f'Analysing date: {YESTERDAY}')
print(f'Learnings DB: {BASE}/learnings_db.json')
print(f'Prompts loaded: extractor={len(_LEARNING_SYSTEM_PROMPT)}c, curator={len(_LEARNING_CURATOR_PROMPT)}c')

## 3b. Load Dividends + Race Verdict JSONs

In [ ]:
# ── Load race verdict JSONs for yesterday ────────────────────────────────────
import glob

_rv_dir = f'{BASE}/race_verdicts'
_json_files = glob.glob(f'{_rv_dir}/{YESTERDAY}_*.json')
print(f'Found {len(_json_files)} verdict JSON files for {YESTERDAY}')

all_race_verdicts = []  # flat list of race dicts
for _jf in _json_files:
    with open(_jf) as _f:
        _races = json.load(_f)
    all_race_verdicts.extend(_races)
    print(f'  {os.path.basename(_jf)}: {len(_races)} races')

print(f'Total races with verdicts: {len(all_race_verdicts)}')


In [ ]:
# Load dividends — filter to raceIds present in today's verdict JSONs
dividends = pd.read_parquet(f'{BASE}/dividends.parquet').drop_duplicates()

# Collect raceIds from loaded verdict JSONs
_verdict_race_ids = set(
    str(r.get('raceId')) for r in all_race_verdicts if r.get('raceId')
)
print(f'raceIds from verdicts: {len(_verdict_race_ids)}')

# Filter dividends to only those races
dividends['raceId'] = dividends['raceId'].astype(str)
div_yday = dividends[dividends['raceId'].isin(_verdict_race_ids)].copy()
print(f'Dividend rows matched: {len(div_yday)}')
div_yday.head(3)


In [ ]:
# ── Extract winners and placed horses from dividends ─────────────────────────
# winner: dividendType=='G' AND originBetType=='S'
# placed: dividendType=='P' AND originBetType=='S'
_win_div  = div_yday[(div_yday['dividendType'] == 'G') & (div_yday['originBetType'] == 'S')][['raceId', 'combination', 'dividend']]
_plc_div  = div_yday[(div_yday['dividendType'] == 'P') & (div_yday['originBetType'] == 'S')][['raceId', 'combination', 'dividend']]

_win_div.columns  = ['raceId', 'saddle', 'win_dividend']
_plc_div.columns  = ['raceId', 'saddle', 'plc_dividend']

# Normalise types
_win_div['raceId'] = _win_div['raceId'].astype(str)
_win_div['saddle'] = _win_div['saddle'].astype(str)
_plc_div['raceId'] = _plc_div['raceId'].astype(str)
_plc_div['saddle'] = _plc_div['saddle'].astype(str)

print(f'Winners found: {len(_win_div)}')
print(f'Placers found: {len(_plc_div)}')


## 3c. Calculate P&L

In [ ]:
# ── Filter to races that have dividend data ───────────────────────────────────
_dividend_race_ids = set(div_yday['raceId'].astype(str).unique())
_before = len(all_race_verdicts)
all_race_verdicts = [r for r in all_race_verdicts if str(r.get('raceId')) in _dividend_race_ids]
print(f'Races with dividend data: {len(all_race_verdicts)} / {_before} '
      f'({_before - len(all_race_verdicts)} skipped — no dividends)')

# ── Build bets table from race verdicts ───────────────────────────────────────
rows = []
for race in all_race_verdicts:
    _rid       = str(race.get('raceId') or '')
    _nap       = race.get('nap') or {}
    _ew        = race.get('each_way') or {}
    _nap_horse = _nap.get('horse', '')
    _ew_horse  = _ew.get('horse', '')

    # Skip races where verdict generation failed to produce a selection
    if not _nap_horse or not _ew_horse:
        continue

    # saddle is stored directly in nap/each_way dicts (added by PT_Vorarbeiten)
    rows.append({
        'datum':          YESTERDAY,
        'raceId':         _rid,
        'course':         race.get('meeting', ''),
        'race':           race.get('race', ''),
        'nap_horse':      _nap_horse,
        'nap_confidence': _nap.get('confidence'),
        'nap_saddle':     str(_nap.get('saddle') or ''),
        'ew_horse':       _ew_horse,
        'ew_confidence':  _ew.get('confidence'),
        'ew_saddle':      str(_ew.get('saddle') or ''),
    })

_BET_COLS = ['datum','raceId','course','race','nap_horse','nap_confidence',
             'nap_saddle','ew_horse','ew_confidence','ew_saddle']
bets_df = pd.DataFrame(rows, columns=_BET_COLS) if rows else pd.DataFrame(columns=_BET_COLS)
bets_df['raceId'] = bets_df['raceId'].astype(str)
_skipped = len(all_race_verdicts) - len(bets_df)
print(f'Bets loaded: {len(bets_df)} races ({_skipped} skipped — no NAP/EW selection)')
if bets_df.empty:
    print('No verdict JSONs found for yesterday — nothing to analyse.')
bets_df.head(3)

In [ ]:
# ── Join with dividends ───────────────────────────────────────────────────────
bets_df = bets_df.merge(_win_div, left_on=['raceId', 'nap_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'win_dividend': 'nap_win_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

bets_df = bets_df.merge(_plc_div, left_on=['raceId', 'nap_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'plc_dividend': 'nap_plc_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

bets_df = bets_df.merge(_win_div, left_on=['raceId', 'ew_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'win_dividend': 'ew_win_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

bets_df = bets_df.merge(_plc_div, left_on=['raceId', 'ew_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'plc_dividend': 'ew_plc_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

# ── Result flags — win and place tracked separately for each selection ─────────
# nap_plc_result='PLACE' whenever place dividend exists (covers both winners and placed)
bets_df['nap_win_result'] = bets_df['nap_win_div'].notna().map({True: 'WIN',   False: 'LOSS'})
bets_df['nap_plc_result'] = bets_df['nap_plc_div'].notna().map({True: 'PLACE', False: 'LOSS'})
bets_df['ew_win_result']  = bets_df['ew_win_div'].notna().map({True: 'WIN',    False: 'LOSS'})
bets_df['ew_plc_result']  = bets_df['ew_plc_div'].notna().map({True: 'PLACE',  False: 'LOSS'})

# ── P&L: 1 unit staked per leg → 4 bets per race (NAP win, NAP plc, EW win, EW plc)
bets_df['nap_win_pnl'] = bets_df['nap_win_div'].apply(lambda d: round(float(d) - 1, 2) if pd.notna(d) else -1.0)
bets_df['nap_plc_pnl'] = bets_df['nap_plc_div'].apply(lambda d: round(float(d) - 1, 2) if pd.notna(d) else -1.0)
bets_df['ew_win_pnl']  = bets_df['ew_win_div'].apply(lambda d: round(float(d) - 1, 2) if pd.notna(d) else -1.0)
bets_df['ew_plc_pnl']  = bets_df['ew_plc_div'].apply(lambda d: round(float(d) - 1, 2) if pd.notna(d) else -1.0)

# ── Day summary ───────────────────────────────────────────────────────────────
_n = len(bets_df)
if _n > 0:
    _nw = (bets_df['nap_win_result'] == 'WIN').sum()
    _np = (bets_df['nap_plc_result'] == 'PLACE').sum()
    _ew = (bets_df['ew_win_result']  == 'WIN').sum()
    _ep = (bets_df['ew_plc_result']  == 'PLACE').sum()
    _nwp = bets_df['nap_win_pnl'].sum()
    _npp = bets_df['nap_plc_pnl'].sum()
    _ewp = bets_df['ew_win_pnl'].sum()
    _epp = bets_df['ew_plc_pnl'].sum()
    _win_pnl   = _nwp + _ewp
    _plc_pnl   = _npp + _epp
    _total_pnl = _win_pnl + _plc_pnl
    _staked    = _n * 4
    _roi       = _total_pnl / _staked * 100

    print(f'{"":4}  {"Win result":>16}  {"Win P&L":>8}  {"Place result":>16}  {"Place P&L":>9}  {"Combined":>9}')
    print(f'NAP   {_nw:2d}/{_n} ({100*_nw/_n:3.0f}%)  {_nwp:+8.2f}u  {_np:2d}/{_n} ({100*_np/_n:3.0f}%)    {_npp:+9.2f}u  {_nwp+_npp:+9.2f}u')
    print(f'EW    {_ew:2d}/{_n} ({100*_ew/_n:3.0f}%)  {_ewp:+8.2f}u  {_ep:2d}/{_n} ({100*_ep/_n:3.0f}%)    {_epp:+9.2f}u  {_ewp+_epp:+9.2f}u')
    print(f'{"─"*75}')
    print(f'Win P&L: {_win_pnl:+.2f}u   Place P&L: {_plc_pnl:+.2f}u   Combined: {_total_pnl:+.2f}u   ROI: {_roi:+.1f}% ({_staked}u staked)')

bets_df[['datum','course','nap_horse','nap_win_result','nap_win_pnl','nap_plc_result','nap_plc_pnl',
          'ew_horse','ew_win_result','ew_win_pnl','ew_plc_result','ew_plc_pnl']]

In [ ]:
# ── Append to betsMonitor.csv ─────────────────────────────────────────────────
_monitor_path = f'{BASE}/betsMonitor.csv'
_cols = ['datum','raceId','course','race',
         'nap_horse','nap_confidence','nap_win_result','nap_win_pnl','nap_plc_result','nap_plc_pnl',
         'ew_horse','ew_confidence','ew_win_result','ew_win_pnl','ew_plc_result','ew_plc_pnl']
_out = bets_df[[c for c in _cols if c in bets_df.columns]].copy()

if os.path.exists(_monitor_path):
    _existing = pd.read_csv(_monitor_path)
    # Handle legacy: 'date' column → 'datum'
    if 'date' in _existing.columns and 'datum' not in _existing.columns:
        _existing = _existing.rename(columns={'date': 'datum'})
    # Handle legacy: old combined column names → new split names (best-effort)
    if 'nap_result' in _existing.columns and 'nap_win_result' not in _existing.columns:
        _existing['nap_win_result'] = _existing['nap_result']
    if 'nap_pnl' in _existing.columns and 'nap_win_pnl' not in _existing.columns:
        _existing['nap_win_pnl'] = _existing['nap_pnl']   # was 1-unit win — maps directly
    if 'ew_result' in _existing.columns and 'ew_win_result' not in _existing.columns:
        _erm = {'WIN': 'WIN', 'PLACE': 'LOSS', 'LOSS': 'LOSS'}
        _epm = {'WIN': 'PLACE', 'PLACE': 'PLACE', 'LOSS': 'LOSS'}
        _existing['ew_win_result'] = _existing['ew_result'].map(_erm)
        _existing['ew_plc_result'] = _existing['ew_result'].map(_epm)
    # ew_pnl was combined 2-unit E/W — can't split retroactively; new columns stay NaN
    if 'datum' in _existing.columns:
        _existing = _existing[_existing['datum'] != YESTERDAY]
    _combined = pd.concat([_existing, _out], ignore_index=True)
else:
    _combined = _out

_combined.to_csv(_monitor_path, index=False)
print(f'✅ Saved betsMonitor.csv ({len(_combined)} total rows, {len(_out)} new for {YESTERDAY})')

# ── All-time aggregate stats ───────────────────────────────────────────────────
_c  = _combined
_cn = len(_c)
if _cn > 0:
    def _sr(col, val):
        return int((_c[col] == val).sum()) if col in _c.columns else 0
    def _pnl(col):
        return float(_c[col].fillna(0).sum()) if col in _c.columns else 0.0

    _nw = _sr('nap_win_result', 'WIN');  _nwp = _pnl('nap_win_pnl')
    _np = _sr('nap_plc_result', 'PLACE'); _npp = _pnl('nap_plc_pnl')
    _ew = _sr('ew_win_result',  'WIN');  _ewp = _pnl('ew_win_pnl')
    _ep = _sr('ew_plc_result',  'PLACE'); _epp = _pnl('ew_plc_pnl')

    _win_pnl   = _nwp + _ewp
    _plc_pnl   = _npp + _epp
    _total_pnl = _win_pnl + _plc_pnl
    _staked    = _cn * 4
    _roi       = _total_pnl / _staked * 100 if _staked > 0 else 0.0

    print(f'\n── All-time aggregate ({_cn} races, {_staked}u staked) ──────────────────────────')
    print(f'{"":4}  {"Win SR":>14}  {"Win P&L":>8}  {"Place SR":>14}  {"Place P&L":>10}  {"Combined":>10}')
    print(f'NAP   {_nw:3d}/{_cn} ({100*_nw/_cn:4.1f}%)  {_nwp:+8.2f}u  {_np:3d}/{_cn} ({100*_np/_cn:4.1f}%)  {_npp:+10.2f}u  {_nwp+_npp:+10.2f}u')
    print(f'EW    {_ew:3d}/{_cn} ({100*_ew/_cn:4.1f}%)  {_ewp:+8.2f}u  {_ep:3d}/{_cn} ({100*_ep/_cn:4.1f}%)  {_epp:+10.2f}u  {_ewp+_epp:+10.2f}u')
    print(f'{"─"*78}')
    print(f'Win P&L: {_win_pnl:+.2f}u   Place P&L: {_plc_pnl:+.2f}u   Combined: {_total_pnl:+.2f}u   ROI: {_roi:+.1f}%')

## 3d. Generate Learnings via Claude

In [ ]:
# ── Enrich race_verdict JSONs with actual results ────────────────────────────
# Each entry = full race_verdict (horses, nap, each_way, going, distance, etc.)
# plus a 'result' block with what actually happened.
enriched_races = []
for race in all_race_verdicts:
    _rid = str(race.get('raceId') or '')
    _row = bets_df[bets_df['raceId'] == _rid]
    if _row.empty:
        continue
    _r = _row.iloc[0]

    # Resolve actual winner name from dividend saddle → horse name
    _winner_row    = _win_div[_win_div['raceId'] == _rid]
    _winner_saddle = _winner_row['saddle'].iloc[0] if not _winner_row.empty else None
    _winner_name   = next(
        (h['name'] for h in race.get('horses', []) if str(h.get('saddle')) == str(_winner_saddle)),
        None
    ) if _winner_saddle else None

    _entry = dict(race)  # full race_verdict JSON — horses, nap, each_way, going, distance, etc.
    _entry['result'] = {
        'nap_win_result': str(_r.get('nap_win_result', '')),
        'nap_plc_result': str(_r.get('nap_plc_result', '')),
        'nap_win_pnl':    round(float(_r.get('nap_win_pnl', 0)), 2),
        'nap_plc_pnl':    round(float(_r.get('nap_plc_pnl', 0)), 2),
        'ew_win_result':  str(_r.get('ew_win_result', '')),
        'ew_plc_result':  str(_r.get('ew_plc_result', '')),
        'ew_win_pnl':     round(float(_r.get('ew_win_pnl', 0)), 2),
        'ew_plc_pnl':     round(float(_r.get('ew_plc_pnl', 0)), 2),
        'actual_winner':  _winner_name,
    }
    enriched_races.append(_entry)

print(f'{len(enriched_races)} enriched races ready for Claude')

In [ ]:
# ── Call 1: extract learnings race by race ───────────────────────────────────
_client       = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
new_learnings = []

# System prompt as cached block — saves tokens from race 2 onward
_learning_system_block = [{'type': 'text', 'text': _LEARNING_SYSTEM_PROMPT,
                            'cache_control': {'type': 'ephemeral'}}]

if not enriched_races:
    print('No race data — skipping learnings extraction.')
else:
    for _i, _race in enumerate(enriched_races):
        _label = f"{_race.get('meeting','?')} — {_race.get('race','?')}"
        try:
            _resp = _client.messages.create(
                model      = 'claude-sonnet-4-6',
                max_tokens = 1024,
                system     = _learning_system_block,
                messages   = [{'role': 'user', 'content':
                    f'Date: {YESTERDAY}\nRace {_i+1}/{len(enriched_races)}: {_label}\n'
                    + json.dumps(_race, indent=2, default=str)
                }],
            )
            if _resp.stop_reason == 'max_tokens':
                print(f'  ⚠️  TOKEN LIMIT: {_label} '
                      f'(in={_resp.usage.input_tokens}, out={_resp.usage.output_tokens})')

            _raw   = re.sub(r'^```(?:json)?\s*', '', _resp.content[0].text.strip())
            _raw   = re.sub(r'\s*```$', '', _raw)
            _match = re.search(r'\[\s*(?:\{[\s\S]*\})?\s*\]', _raw)
            if _match:
                _race_learnings = json.loads(_match.group())
                new_learnings.extend(_race_learnings)
                if _race_learnings:
                    _cache_hit = getattr(_resp.usage, 'cache_read_input_tokens', 0) or 0
                    print(f'  [{_i+1}/{len(enriched_races)}] {_label}: '
                          f'{len(_race_learnings)} learnings  '
                          f'(in={_resp.usage.input_tokens}t'
                          + (f', cache_hit={_cache_hit}t' if _cache_hit else '') + ')')
            else:
                print(f'  [{_i+1}/{len(enriched_races)}] {_label}: no learnings')

        except Exception as _e:
            print(f'  ⚠️  {_label}: API error — {_e}')

    print(f'\nTotal new learnings from all races: {len(new_learnings)}')
    for _l in new_learnings:
        print(f'  [{_l.get("direction")}] {_l.get("id")}: {_l.get("learning")}')

## 3e. Merge into DB + Curate

In [ ]:
# ── Load existing DB and merge today's new learnings ─────────────────────────
_db_path = f'{BASE}/learnings_db.json'
learnings_db = []
if os.path.exists(_db_path):
    with open(_db_path) as _f:
        learnings_db = json.load(_f)
print(f'Existing learnings: {len(learnings_db)}')

_existing_ids = {l['id']: i for i, l in enumerate(learnings_db)}
for _nl in new_learnings:
    _nid = _nl.get('id', '')
    if _nid in _existing_ids:
        _idx = _existing_ids[_nid]
        learnings_db[_idx]['counter'] = learnings_db[_idx].get('counter', 1) + 1
        learnings_db[_idx]['last_updated'] = YESTERDAY
        print(f'  Updated: {_nid} (counter={learnings_db[_idx]["counter"]})')
    else:
        _nl.setdefault('counter', 1)
        _nl['last_updated'] = YESTERDAY
        learnings_db.append(_nl)
        _existing_ids[_nid] = len(learnings_db) - 1
        print(f'  Added:   {_nid}')

print(f'DB size after merge: {len(learnings_db)}')

# ── Call 2: curate the full DB (generalize, merge, condense, remove) ─────────
if learnings_db and enriched_races:
    _resp2 = _client.messages.create(
        model      = 'claude-sonnet-4-6',
        max_tokens = 4096,
        system     = _LEARNING_CURATOR_PROMPT,
        messages   = [{'role': 'user', 'content':
            f'Today is {YESTERDAY}. Curate this learning database:\n'
            + json.dumps(learnings_db, indent=2, default=str)
        }],
    )
    if _resp2.stop_reason == 'max_tokens':
        print(
            f'⚠️  TOKEN LIMIT: learning curator truncated '
            f'(max_tokens=4096, in={_resp2.usage.input_tokens}, '
            f'out={_resp2.usage.output_tokens}) — increase max_tokens'
        )

    _raw2   = re.sub(r'^```(?:json)?\s*', '', _resp2.content[0].text.strip())
    _raw2   = re.sub(r'\s*```$', '', _raw2)
    _match2 = re.search(r'\[\s*(?:\{[\s\S]*\})?\s*\]', _raw2)
    if _match2:
        try:
            _curated = json.loads(_match2.group())
            if _curated:
                _prev_size   = len(learnings_db)
                learnings_db = _curated
                print(f'Curated: {_prev_size} → {len(learnings_db)} learnings '
                      f'(in={_resp2.usage.input_tokens} / out={_resp2.usage.output_tokens} tokens)')
            else:
                print('Curator returned empty list — keeping current DB')
        except json.JSONDecodeError as _e:
            print(f'Curator JSON parse error: {_e} — keeping uncurated DB')
    else:
        print('Curator returned no valid JSON — keeping uncurated DB')
else:
    learnings_db.sort(key=lambda x: x.get('counter', 0), reverse=True)

# ── Save ──────────────────────────────────────────────────────────────────────
with open(_db_path, 'w', encoding='utf-8') as _f:
    json.dump(learnings_db, _f, ensure_ascii=False, indent=2)

print(f'\n✅ learnings_db.json saved: {len(learnings_db)} learnings')
for _l in learnings_db[:5]:
    print(f'   [{_l.get("counter",1)}x] {_l.get("id")}: {_l.get("learning","")[:80]}')

## Summary

In [ ]:
print('\n✅ PT Monitor & Learning complete')
print(f'   Date analysed:    {YESTERDAY}')
print(f'   Races processed:  {len(all_race_verdicts)}')
print(f'   betsMonitor.csv:  {BASE}/betsMonitor.csv')
print(f'   learnings_db.json:{BASE}/learnings_db.json')
